In [1]:
import numpy as np
from scipy.special import iv  # Modified Bessel functions

# SU(3) fundamental representations by Young tableaux
# (p,q) where p = boxes in first row, q = boxes in second row
irreps = [
    (0, 0),  # trivial (singlet)
    (1, 0),  # fundamental (3)
    (0, 1),  # anti-fundamental (3*)
    (2, 0),  # symmetric (6)
    (1, 1),  # adjoint (8)
    (0, 2),  # anti-symmetric (6*)
    (3, 0),  # (10)
    (2, 1),  # (15)
    # ... add more as needed
]

def su3_dimension(p, q):
    """Dimension of SU(3) irrep (p,q)"""
    return ((p + 1) * (q + 1) * (p + q + 2)) // 2

def character_coefficient(p, q, beta):
    """Coefficient c_{p,q}(β) in character expansion"""
    d = su3_dimension(p, q)
    # For SU(3): c_{p,q}(β) = I_{p+q}(β) * polynomial_factor(p,q,β)
    return iv(p + q, beta) * bessel_polynomial_factor(p, q, beta)

def bessel_polynomial_factor(p, q, beta):
    """Additional polynomial factors for SU(3) characters"""
    # This is the SU(3)-specific correction to the Bessel function
    # Exact formula depends on representation theory
    if (p, q) == (0, 0):
        return 1.0
    elif (p, q) == (1, 0) or (p, q) == (0, 1):
        return beta / 3.0
    elif (p, q) == (1, 1):
        return beta**2 / 9.0
    else:
        # Higher order terms - implement as needed
        return (beta**(p+q)) / (3**(p+q))


In [4]:
def verify_character_expansion(beta, tolerance=1e-10):
    """Verify that exp(β Re Tr U) = Σ c_r(β) χ_r(U)"""

    # Generate random SU(3) matrix
    U = random_su3_matrix()

    # Direct calculation
    direct = np.exp(beta * np.real(np.trace(U)) / 3)

    # Character expansion
    expansion = 0.0
    for p, q in irreps[:10]:  # Use first 10 irreps
        c_pq = character_coefficient(p, q, beta)
        chi_pq = su3_character(U, p, q)
        expansion += c_pq * chi_pq

        if c_pq < 0:
            raise ValueError(f"Negative coefficient c_{p},{q} = {c_pq}")

    error = abs(direct - expansion)
    print(f"β={beta}: Direct={direct:.6f}, Expansion={expansion:.6f}, Error={error:.2e}")

    return error < tolerance

# Test it
for beta in [0.1, 0.5, 1.0, 2.0]:
    verify_character_expansion(beta)


β=0.1: Direct=0.985040, Expansion=-0.502922, Error=1.49e+00
β=0.5: Direct=0.858958, Expansion=-0.576024, Error=1.43e+00
β=1.0: Direct=1.348928, Expansion=-0.874376, Error=2.22e+00
β=2.0: Direct=0.585597, Expansion=-2.737328, Error=3.32e+00


In [3]:
def su3_character(U, p, q):
    """Compute character χ_{p,q}(U) for SU(3) matrix U"""
    eigenvals = np.linalg.eigvals(U)
    # Sort eigenvalues: e^{iθ₁}, e^{iθ₂}, e^{iθ₃}
    theta = np.angle(eigenvals)

    # SU(3) character formula using Weyl character formula
    return weyl_character_su3(theta[0], theta[1], theta[2], p, q)

def weyl_character_su3(theta1, theta2, theta3, p, q):
    """Weyl character formula for SU(3)"""
    # Verify constraint: θ₁ + θ₂ + θ₃ = 0 (mod 2π)

    numerator = (
        np.exp(1j * ((p+1)*theta1 + (q+1)*theta2)) +
        np.exp(1j * ((p+1)*theta2 + (q+1)*theta3)) +
        np.exp(1j * ((p+1)*theta3 + (q+1)*theta1)) -
        np.exp(1j * (p*theta1 + q*theta2)) -
        np.exp(1j * (p*theta2 + q*theta3)) -
        np.exp(1j * (p*theta3 + q*theta1))
    )

    denominator = (
        np.exp(1j * theta1) + np.exp(1j * theta2) + np.exp(1j * theta3) -
        np.exp(-1j * theta1) - np.exp(-1j * theta2) - np.exp(-1j * theta3)
    )

    return np.real(numerator / denominator)

def random_su3_matrix():
    """Generate random SU(3) matrix using Haar measure"""
    # Use QR decomposition method for Haar-distributed SU(3)
    A = np.random.randn(3, 3) + 1j * np.random.randn(3, 3)
    Q, R = np.linalg.qr(A)
    # Normalize to get SU(3)
    Q = Q / (np.linalg.det(Q)**(1/3))
    return Q
